In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm

# Paths
pose_dir = os.path.expanduser("~/badminton/features/pose")
shuttle_dir = os.path.expanduser("~/badminton/features/shuttle")
output_dir = os.path.expanduser("~/badminton/clip_features")
os.makedirs(output_dir, exist_ok=True)

# Helper: load YOLO shuttle positions per frame
def load_shuttle_positions(label_path, num_frames):
    shuttle_xy = np.full((num_frames, 2), -1.0)
    if not os.path.exists(label_path):
        return shuttle_xy

    label_files = sorted(os.listdir(label_path))
    for i, fname in enumerate(label_files):
        with open(os.path.join(label_path, fname), 'r') as f:
            lines = f.readlines()
            if lines:
                parts = lines[0].strip().split()
                if len(parts) == 5:
                    x = float(parts[1])
                    y = float(parts[2])
                    shuttle_xy[i] = [x, y]
    return shuttle_xy

# Main processing loop
for fname in tqdm(os.listdir(pose_dir)):
    if not fname.endswith("_pose.json"):
        continue

    base = fname.replace("_pose.json", "")
    pose_path = os.path.join(pose_dir, fname)
    label_path = os.path.join(shuttle_dir, base)  # dir of txts

    # Load pose
    with open(pose_path, 'r') as f:
        pose_data = json.load(f)

    frames = pose_data.get("frames", [])
    num_frames = len(frames)
    pose_array = np.zeros((num_frames, 34))  # 17 keypoints x (x, y)

    for i, frame in enumerate(frames):
        keypoints = frame.get("keypoints", [])
        if len(keypoints) >= 17:
            for j in range(17):
                kp = keypoints[j]
                pose_array[i, j*2] = kp.get("x", 0.0)
                pose_array[i, j*2+1] = kp.get("y", 0.0)

    # Load shuttle positions
    shuttle_array = load_shuttle_positions(label_path, num_frames)

    # Combine [T, 34] + [T, 2] = [T, 36]
    feature_array = np.concatenate([pose_array, shuttle_array], axis=1)
    
    # Save
    out_path = os.path.join(output_dir, f"{base}_features.npy")
    np.save(out_path, feature_array)

    print(f"Saved features for {base} → {out_path}")


In [1]:
raw_data = """
1 
356
427
1
612
982
1
1463
1814
2
2032
2167
2
2398
2697
2
2941
3128
2
3384
3541
2
3376
3814
2
4035
4155
1
4339
4948
2
5172
5382
2
5621
5799
1
6036
6202
1
6411
6604
2
6884
6999
1
7228
7338
2
7645
7918
2
8210
8263
1
8469
8743
2
8971
9128
2
9346
9451
1
9722
9844

"""

# Split and filter
tokens = raw_data.split()
rally_numbers = [int(x) for x in tokens if x not in {"1", "2"}]

# Group into (start, end) pairs
rally_intervals = list(zip(rally_numbers[::2], rally_numbers[1::2]))

# Output
for start, end in rally_intervals:
    print(f"({start}, {end})")

# Optionally store in a list variable
print("\nAs a Python list:")
print("rally_intervals = [")
for start, end in rally_intervals:
    print(f"    ({start}, {end}),")
print("]")


(356, 427)
(612, 982)
(1463, 1814)
(2032, 2167)
(2398, 2697)
(2941, 3128)
(3384, 3541)
(3376, 3814)
(4035, 4155)
(4339, 4948)
(5172, 5382)
(5621, 5799)
(6036, 6202)
(6411, 6604)
(6884, 6999)
(7228, 7338)
(7645, 7918)
(8210, 8263)
(8469, 8743)
(8971, 9128)
(9346, 9451)
(9722, 9844)

As a Python list:
rally_intervals = [
    (356, 427),
    (612, 982),
    (1463, 1814),
    (2032, 2167),
    (2398, 2697),
    (2941, 3128),
    (3384, 3541),
    (3376, 3814),
    (4035, 4155),
    (4339, 4948),
    (5172, 5382),
    (5621, 5799),
    (6036, 6202),
    (6411, 6604),
    (6884, 6999),
    (7228, 7338),
    (7645, 7918),
    (8210, 8263),
    (8469, 8743),
    (8971, 9128),
    (9346, 9451),
    (9722, 9844),
]
